In [69]:
pd.set_option('display.max_columns', None)

In [1]:
import pandas as pd
import numpy as np

Here we will explore how we can leverage LLMs to improve or replace traditional NLP techniques

In [100]:
df_llm = pd.read_csv("data/wine_eda_features.csv")
df_llm.head()

,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.500000,0.246407,0.335451,1.000000
1,France,A very herbaceous character makes this wine se...,0.277032,0.222222,0.293354,0.272456,0.076923
2,US,"A little simple and easy, but there's a wealth...",0.265230,0.222222,0.309569,0.260956,0.740385
3,US,From a dry-farmed vineyard and treated to wild...,0.424722,0.444444,0.155139,0.426811,0.586538
4,US,"From a site near Annapolis, this wine shows a ...",0.537366,0.611111,0.092931,0.544124,0.634615


# LLM leveraging

LLM can significantly improve or even replace traditional NLP techniques. Traditional methods (NLP) treat words independently and ignore context, while LLMs capture contextual relationships. Leveraging LLMs is useful in terms of efficiency, since they are pre-trained on massive datasets. Further to that it can optimise our features since it can handle several tasks at once - such as text classification, NER and semantic similarity all within one framework. Therefore it generally replaces the need for separate feature engineering (could have saved time on all of the NLP features being engineered separately)

I will now utilise the provided API for the following steps:
1. extracting semantic features - generate embeddings and summaries from wine descriptions
2. NER - use LLM to extract locations, grape varieties or regions
3. sentiment analysis - whether certain countries have more positively described wines

## API set-up

In [3]:
import requests
API_KEY = 'YspgxAulhv3CQnaLm6d4M4dPkboijSYYXksxln8c'
API_URL = 'https://candidate-llm.extraction.artificialos.com/v1/chat/completions'

## Extracting semantic summaries and attributes

We will query gpt-4o-mini model for it to generate key attributes for wine from descriptions. For example we can later transform those attributes into binary features (has_cherry, has_oak) or categorical features (tanin_level - may be high or low for example)

The model sometimes throws timeout errors, those are taken care of below

In [110]:
import time

def extract_attributes_with_retries(description, max_retries=5, initial_wait=2):
    headers = {
        'Content-Type': 'application/json',
        'x-api-key': API_KEY
    }

    payload = {
        "model": "gpt-4o-mini",
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert sommelier. Extract only the key attributes from the wine description in the following JSON format: "
                    "{ \"Flavor\": [list of flavors], \"Body\": [light/medium/full], \"Tannins\": [low/medium/high], "
                    "\"Sweetness\": [dry/sweet/very sweet], \"Acidity\": [low/medium/high], \"Finish\": [short/medium/long] }. "
                    "You may include other attributes that you find relevant. Put characterististics like 'Ageability', 'Aging Potential', 'Ageworthiness', 'Ageing Potential' into one parameter named \"Ageability\": [low/medium/high]."
                    "Do not include explanations."
                )
            },
            {"role": "user", "content": f"Description: {description}"}
        ],
        "temperature": 0
    }

    wait_time = initial_wait
    retries = 0

    while retries < max_retries:
        try:
            response = requests.post(API_URL, headers=headers, json=payload, timeout=30)  # Timeout for slow responses

            if response.status_code == 200:
                # Successful response
                return response.json()['choices'][0]['message']['content'].strip()

            elif response.status_code == 429:
                # Rate limit error
                print(f"Rate limit hit (429). Retrying in {wait_time} seconds... (Attempt {retries + 1}/{max_retries})")
                time.sleep(wait_time)
                retries += 1
                wait_time *= 2 

            elif response.status_code == 504:
                # Gateway timeout error
                print(f"Gateway timeout (504). Retrying in {wait_time} seconds... (Attempt {retries + 1}/{max_retries})")
                time.sleep(wait_time)
                retries += 1
                wait_time *= 2 

            else:
                # Other errors
                return f"Error: {response.status_code}"

        except requests.exceptions.Timeout:
            # Handle client-side timeout
            print(f"Client-side timeout. Retrying in {wait_time} seconds... (Attempt {retries + 1}/{max_retries})")
            time.sleep(wait_time)
            retries += 1
            wait_time *= 2  

    # If max retries exceeded
    print(f"Failed after {max_retries} retries.")
    return "Error: Max retries exceeded"

# Apply the function with delay
df= df_llm.copy(deep=True)
df['extracted_attributes'] = df['description'].apply(extract_attributes_with_retries)

Rate limit hit (429). Retrying in 2 seconds... (Attempt 1/5)
Rate limit hit (429). Retrying in 2 seconds... (Attempt 1/5)
Rate limit hit (429). Retrying in 2 seconds... (Attempt 1/5)
Rate limit hit (429). Retrying in 2 seconds... (Attempt 1/5)
Rate limit hit (429). Retrying in 2 seconds... (Attempt 1/5)
Rate limit hit (429). Retrying in 2 seconds... (Attempt 1/5)
Rate limit hit (429). Retrying in 2 seconds... (Attempt 1/5)
Rate limit hit (429). Retrying in 2 seconds... (Attempt 1/5)


In [111]:
import json
def clean_and_parse_json(string):
    try:
        # # Step 1: Filter out non-JSON responses (like API errors)
        # if not string.strip().startswith('{') or not string.strip().endswith('}'):
        #     raise ValueError("Not a JSON string")

        # Step 2: Replace single quotes with double quotes
        #json_compliant_str = string.replace("'", '"')

        # # Step 3: Remove trailing commas before closing braces or brackets
        # json_compliant_str = re.sub(r',\s*([}\]])', r'\1', json_compliant_str)

        # # Step 4: Remove leading/trailing whitespace
        # json_compliant_str = json_compliant_str.strip()

        # # Step 5: Parse the cleaned string as JSON
        return json.loads(string)

    except (json.JSONDecodeError, ValueError) as e:
        # For debugging: print problematic strings
        print(f"JSON parsing failed: {e}")
        print(f"Problematic string: {string}")
        return None  # Return None if parsing fails

# Apply the cleaning and parsing function
df['attributes_dict'] = df['extracted_attributes'].apply(clean_and_parse_json)

# Display successfully parsed attributes
df[['extracted_attributes', 'attributes_dict']]

,extracted_attributes,attributes_dict
0,"{\n ""Flavor"": [""sour cherry candy"", ""red curr...","{'Flavor': ['sour cherry candy', 'red currants..."
1,"{\n ""Flavor"": [""herbaceous"", ""lemon zest""],\n...","{'Flavor': ['herbaceous', 'lemon zest'], 'Body..."
2,"{\n ""Flavor"": [""raspberry"", ""strawberry"", ""fl...","{'Flavor': ['raspberry', 'strawberry', 'floral..."
3,"{\n ""Flavor"": [""blackberries"", ""coffee"", ""pep...","{'Flavor': ['blackberries', 'coffee', 'pepper'..."
4,"{\n ""Flavor"": [""dark grape"", ""cherry"", ""cinna...","{'Flavor': ['dark grape', 'cherry', 'cinnamon'..."
...,...,...
995,"{\n ""Flavor"": [""white fruit""],\n ""Body"": [""m...","{'Flavor': ['white fruit'], 'Body': ['medium']..."
996,"{\n ""Flavor"": [""green"", ""raisin"", ""gummy""],\n...","{'Flavor': ['green', 'raisin', 'gummy'], 'Body..."
997,"{\n ""Flavor"": [""blackberry"", ""currant""],\n ""...","{'Flavor': ['blackberry', 'currant'], 'Body': ..."
998,"{\n ""Flavor"": [""violets"", ""lemon blossoms"", ""...","{'Flavor': ['violets', 'lemon blossoms', 'mine..."


In [114]:
df = df.drop('extracted_attributes', axis=1)

In [115]:
df.columns

Index(['country', 'description', 'log_price_normalized', 'points_normalized',
       'vfm_normalized', 'log_pxp_normalized', 'encoded_variety',
       'attributes_dict', 'Ageability_0', 'Ageability_1', 'Ageability_2',
       'Ageability_3', 'Acidity_0', 'Acidity_1', 'Acidity_2', 'Acidity_3',
       'Acidity_4', 'Sweetness_0', 'Sweetness_1', 'Sweetness_2', 'Sweetness_3',
       'Sweetness_4', 'Sweetness_5', 'Sweetness_6', 'Sweetness_7',
       'Sweetness_8', 'Sweetness_9', 'Sweetness_10', 'Sweetness_11', 'Body_0',
       'Body_1', 'Body_2', 'Body_3', 'Body_4', 'Body_5', 'Body_6', 'Tannins_0',
       'Tannins_1', 'Tannins_2', 'Tannins_3', 'Tannins_4', 'Finish_0',
       'Finish_1', 'Finish_2', 'Finish_3', 'Finish_4', 'Finish_5', 'Finish_6',
       'Finish_7', 'Finish_8', 'Finish_9'],
      dtype='object')

Now that we have extracted json descriptions from LLM we need to identify unique attributes listed in them and encode them as categorical values (and one-hot encode list attributes)

In [112]:
unique_attributes = set()
df['attributes_dict'].apply(lambda x: unique_attributes.update(x.keys()) if x else None)

print("Unique Attributes Found:", unique_attributes)


Unique Attributes Found: {'Flavor', 'Ageability', 'Acidity', 'Sweetness', 'Body', 'Tannins', 'Finish'}


In [113]:
from sklearn.preprocessing import LabelEncoder

categorical_attributes = ['Ageability', 'Acidity', 'Sweetness', 'Body', 'Tannins', 'Finish']

for attribute in categorical_attributes:
    encoder = LabelEncoder()
    attribute_values = df['attributes_dict'].apply(lambda x: x.get(attribute, 'unknown') if isinstance(x, dict) else 'unknown')
    attribute_values = attribute_values.apply(lambda x: ', '.join(x) if isinstance(x, list) else str(x))
    df[f'{attribute}_Encoded'] = encoder.fit_transform(attribute_values)


encoded_columns = [f'{attribute}_Encoded' for attribute in categorical_attributes]

df = pd.get_dummies(df, columns=encoded_columns, prefix=categorical_attributes)

df.head(3)

,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,extracted_attributes,attributes_dict,Ageability_0,Ageability_1,Ageability_2,Ageability_3,Acidity_0,Acidity_1,Acidity_2,Acidity_3,Acidity_4,Sweetness_0,Sweetness_1,Sweetness_2,Sweetness_3,Sweetness_4,Sweetness_5,Sweetness_6,Sweetness_7,Sweetness_8,Sweetness_9,Sweetness_10,Sweetness_11,Body_0,Body_1,Body_2,Body_3,Body_4,Body_5,Body_6,Tannins_0,Tannins_1,Tannins_2,Tannins_3,Tannins_4,Finish_0,Finish_1,Finish_2,Finish_3,Finish_4,Finish_5,Finish_6,Finish_7,Finish_8,Finish_9
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.500000,0.246407,0.335451,1.000000,"{\n ""Flavor"": [""sour cherry candy"", ""red curr...","{'Flavor': ['sour cherry candy', 'red currants...",False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False
1,France,A very herbaceous character makes this wine se...,0.277032,0.222222,0.293354,0.272456,0.076923,"{\n ""Flavor"": [""herbaceous"", ""lemon zest""],\n...","{'Flavor': ['herbaceous', 'lemon zest'], 'Body...",False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False
2,US,"A little simple and easy, but there's a wealth...",0.265230,0.222222,0.309569,0.260956,0.740385,"{\n ""Flavor"": [""raspberry"", ""strawberry"", ""fl...","{'Flavor': ['raspberry', 'strawberry', 'floral...",False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False


In [116]:
import warnings

# Suppress all warnings globally
warnings.filterwarnings('ignore')

In [130]:
list_attributes = ['Flavor']

# Identify columns generated from list attributes
for attribute in list_attributes:
    df[attribute] = df['attributes_dict'].apply(lambda x: ' '.join(x.get(attribute, [])) if isinstance(x.get(attribute), list) else 'unknown')

# Display the DataFrame to confirm changes
df[list_attributes].head()

,Flavor
0,sour cherry candy red currants blueberries tar...
1,herbaceous lemon zest
2,raspberry strawberry floral
3,blackberries coffee pepper
4,dark grape cherry cinnamon black peppercorn ru...


In [131]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')

# df['Flavor_str'] = df['Flavor'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

# # Step 2: Tokenize the stringified 'Flavor' column
# df['tokenized_flavor'] = df['Flavor_str'].apply(lambda x: word_tokenize(x.lower()))
df['Flavor'] = df['Flavor'].apply(lambda x: x.lower().split())
non_empty_sentences = df['Flavor'].tolist()

#Word2Vec.build_vocab(non_empty_sentences, update=False)
#sentences = df['tokenized_flavor'].tolist()
word2vec_model = Word2Vec(sentences=non_empty_sentences, vector_size=50, window=3, min_count=1, workers=4, seed=42)

def average_embeddings(words, model, vector_size):
    if not words:
        return np.zeros(vector_size)
    vectors = [model.wv[word] for word in words if word in model.wv]
    if not vectors:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

for attribute in list_attributes:
    embedding_columns = [f'{attribute}_embed_{i}' for i in range(word2vec_model.vector_size)]
    df[embedding_columns] = df[attribute].apply(lambda x: pd.Series(average_embeddings(x, word2vec_model, word2vec_model.vector_size)))

df.head()

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vgasp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,attributes_dict,Ageability_0,Ageability_1,Ageability_2,Ageability_3,Acidity_0,Acidity_1,Acidity_2,Acidity_3,Acidity_4,Sweetness_0,Sweetness_1,Sweetness_2,Sweetness_3,Sweetness_4,Sweetness_5,Sweetness_6,Sweetness_7,Sweetness_8,Sweetness_9,Sweetness_10,Sweetness_11,Body_0,Body_1,Body_2,Body_3,Body_4,Body_5,Body_6,Tannins_0,Tannins_1,Tannins_2,Tannins_3,Tannins_4,Finish_0,Finish_1,Finish_2,Finish_3,Finish_4,Finish_5,Finish_6,Finish_7,Finish_8,Finish_9,Flavor,Flavor_str,tokenized_flavor,Flavor_embed_0,Flavor_embed_1,Flavor_embed_2,Flavor_embed_3,Flavor_embed_4,Flavor_embed_5,Flavor_embed_6,Flavor_embed_7,Flavor_embed_8,Flavor_embed_9,Flavor_embed_10,Flavor_embed_11,Flavor_embed_12,Flavor_embed_13,Flavor_embed_14,Flavor_embed_15,Flavor_embed_16,Flavor_embed_17,Flavor_embed_18,Flavor_embed_19,Flavor_embed_20,Flavor_embed_21,Flavor_embed_22,Flavor_embed_23,Flavor_embed_24,Flavor_embed_25,Flavor_embed_26,Flavor_embed_27,Flavor_embed_28,Flavor_embed_29,Flavor_embed_30,Flavor_embed_31,Flavor_embed_32,Flavor_embed_33,Flavor_embed_34,Flavor_embed_35,Flavor_embed_36,Flavor_embed_37,Flavor_embed_38,Flavor_embed_39,Flavor_embed_40,Flavor_embed_41,Flavor_embed_42,Flavor_embed_43,Flavor_embed_44,Flavor_embed_45,Flavor_embed_46,Flavor_embed_47,Flavor_embed_48,Flavor_embed_49
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.500000,0.246407,0.335451,1.000000,"{'Flavor': ['sour cherry candy', 'red currants...",False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,"[sour, cherry, candy, red, currants, blueberri...",,[],0.002942,-0.018224,-0.008110,-0.004448,0.005891,0.005300,-0.006271,0.024145,-0.025567,0.000539,-0.013274,0.016820,0.002571,0.014558,0.006142,-0.007492,-0.002435,-0.025775,-0.011514,-0.005155,-0.001678,0.010637,0.009606,0.001790,0.014124,0.002587,-0.003902,-0.001312,-0.003493,-0.015485,-0.002237,-0.008857,-0.009433,0.008766,-0.007805,-0.008802,0.002012,-0.037874,-0.006663,0.013167,-0.010712,0.006425,0.024395,-0.023538,0.001811,-0.003636,0.002449,0.013644,0.021846,-0.009787
1,France,A very herbaceous character makes this wine se...,0.277032,0.222222,0.293354,0.272456,0.076923,"{'Flavor': ['herbaceous', 'lemon zest'], 'Body...",False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,"[herbaceous, lemon, zest]",,[],0.004058,-0.021476,-0.000690,0.000125,0.005485,0.002280,0.006493,0.018077,-0.012990,-0.004130,-0.020144,0.004658,0.004603,0.009171,0.004392,0.003040,0.001215,-0.025576,-0.019305,-0.003008,0.004934,0.017713,0.002444,0.008406,0.009515,-0.001299,-0.005225,-0.004177,-0.021672,-0.003125,-0.016052,-0.009509,-0.003281,-0.005493,0.001079,0.002157,-0.004766,-0.028993,0.012287,0.016535,-0.010186,0.007179,0.025674,-0.006995,-0.005597,-0.010680,0.010852,0.019176,0.018268,-0.007788
2,US,"A little simple and easy, but there's a wealth...",0.265230,0.222222,0.309569,0.260956,0.740385,"{'Flavor': ['raspberry', 'strawberry', 'floral...",False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,"[raspberry, strawberry, floral]",,[],0.007675,-0.024040,0.001876,0.008715,0.012410,-0.004443,-0.001269,0.014503,-0.009059,0.008798,-0.019178,0.001145,0.002452,0.002128,0.002907,-0.016415,-0.008019,-0.016088,-0.004900,-0.005635,-0.001141,0.001355,0.009549,-0.005562,0.007681,0.002801,-0.007487,-0.000338,-0.012366,-0.017702,-0.005149,0.007899,-0.009086,

In [ ]:
df = df.drop('extracted_attributes', axis=1)

In [137]:
df.head(3)

,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,Ageability_0,Ageability_1,Ageability_2,Ageability_3,Acidity_0,Acidity_1,Acidity_2,Acidity_3,Acidity_4,Sweetness_0,Sweetness_1,Sweetness_2,Sweetness_3,Sweetness_4,Sweetness_5,Sweetness_6,Sweetness_7,Sweetness_8,Sweetness_9,Sweetness_10,Sweetness_11,Body_0,Body_1,Body_2,Body_3,Body_4,Body_5,Body_6,Tannins_0,Tannins_1,Tannins_2,Tannins_3,Tannins_4,Finish_0,Finish_1,Finish_2,Finish_3,Finish_4,Finish_5,Finish_6,Finish_7,Finish_8,Finish_9,Flavor_embed_0,Flavor_embed_1,Flavor_embed_2,Flavor_embed_3,Flavor_embed_4,Flavor_embed_5,Flavor_embed_6,Flavor_embed_7,Flavor_embed_8,Flavor_embed_9,Flavor_embed_10,Flavor_embed_11,Flavor_embed_12,Flavor_embed_13,Flavor_embed_14,Flavor_embed_15,Flavor_embed_16,Flavor_embed_17,Flavor_embed_18,Flavor_embed_19,Flavor_embed_20,Flavor_embed_21,Flavor_embed_22,Flavor_embed_23,Flavor_embed_24,Flavor_embed_25,Flavor_embed_26,Flavor_embed_27,Flavor_embed_28,Flavor_embed_29,Flavor_embed_30,Flavor_embed_31,Flavor_embed_32,Flavor_embed_33,Flavor_embed_34,Flavor_embed_35,Flavor_embed_36,Flavor_embed_37,Flavor_embed_38,Flavor_embed_39,Flavor_embed_40,Flavor_embed_41,Flavor_embed_42,Flavor_embed_43,Flavor_embed_44,Flavor_embed_45,Flavor_embed_46,Flavor_embed_47,Flavor_embed_48,Flavor_embed_49
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.500000,0.246407,0.335451,1.000000,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,0.002942,-0.018224,-0.008110,-0.004448,0.005891,0.005300,-0.006271,0.024145,-0.025567,0.000539,-0.013274,0.016820,0.002571,0.014558,0.006142,-0.007492,-0.002435,-0.025775,-0.011514,-0.005155,-0.001678,0.010637,0.009606,0.001790,0.014124,0.002587,-0.003902,-0.001312,-0.003493,-0.015485,-0.002237,-0.008857,-0.009433,0.008766,-0.007805,-0.008802,0.002012,-0.037874,-0.006663,0.013167,-0.010712,0.006425,0.024395,-0.023538,0.001811,-0.003636,0.002449,0.013644,0.021846,-0.009787
1,France,A very herbaceous character makes this wine se...,0.277032,0.222222,0.293354,0.272456,0.076923,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,0.004058,-0.021476,-0.000690,0.000125,0.005485,0.002280,0.006493,0.018077,-0.012990,-0.004130,-0.020144,0.004658,0.004603,0.009171,0.004392,0.003040,0.001215,-0.025576,-0.019305,-0.003008,0.004934,0.017713,0.002444,0.008406,0.009515,-0.001299,-0.005225,-0.004177,-0.021672,-0.003125,-0.016052,-0.009509,-0.003281,-0.005493,0.001079,0.002157,-0.004766,-0.028993,0.012287,0.016535,-0.010186,0.007179,0.025674,-0.006995,-0.005597,-0.010680,0.010852,0.019176,0.018268,-0.007788
2,US,"A little simple and easy, but there's a wealth...",0.265230,0.222222,0.309569,0.260956,0.740385,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,0.007675,-0.024040,0.001876,0.008715,0.012410,-0.004443,-0.001269,0.014503,-0.009059,0.008798,-0.019178,0.001145,0.002452,0.002128,0.002907,-0.016415,-0.008019,-0.016088,-0.004900,-0.005635,-0.001141,0.001355,0.009549,-0.005562,0.007681,0.002801,-0.007487,-0.000338,-0.012366,-0.017702,-0.005149,0.007899,-0.009086,-0.000200,-0.002899,0.002067,-0.003442,-0.031133,-0.004526,0.011214,-0.004241,0.002894,0.017400,-0.004891,0.006391,-0.006254,0.008617,0.016128,0.014804,-0.006356


Since LLM approach generated too many columns, we can remove those that have a threshold of 99% 0 values

## LLM powered NER

We will now go through extracting regions and wine locations. We have done similar work in the previous notebook and now will see how does llm apply to this task

In [155]:
def extract_dynamic_locations(description, retries=3, wait=5):
    headers = {
        'Content-Type': 'application/json',
        'x-api-key': API_KEY
    }

    payload = {
        "model": "gpt-4o-mini",
        "messages": [
            {
                "role": "system",
                "content": (
                    "Extract location-related information from the following wine description. "
                    "Come up with 4 factors that describe location . "
                    "Return everything in a JSON format like the following:"
                    "{ \"Region\": [\"California\", \"Annapolis\" etc] }. if the attribute is not present in description, do not include them in JSON for that description. "
                )
            },
            {"role": "user", "content": f"Description: {description}"}
        ],
        "temperature": 0
    }

    for attempt in range(retries):
        response = requests.post(API_URL, headers=headers, json=payload)
        
        if response.status_code == 200:
            # Return the raw JSON content as string
            result = response.json()
            raw_json_string = result['choices'][0]['message']['content'].strip()
            return raw_json_string
        
        elif response.status_code in [429, 504]:  # Handle rate limits or timeouts
            print(f"Error {response.status_code} on attempt {attempt + 1}. Retrying in {wait} seconds...")
            time.sleep(wait)
        else:
            print(f"Unexpected error {response.status_code}: {response.text}")
            return None

    return "Error: Max retries exceeded"

def_ner = df_llm.copy(deep=True)

def_ner['extracted_locations'] = def_ner['description'].apply(extract_dynamic_locations)
def_ner[['description', 'extracted_locations']].head()

Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 2. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt

,description,extracted_locations
0,"Very good Dry Creek Zin, robust and dry and sp...","{\n ""Region"": [""Dry Creek""]\n}"
1,A very herbaceous character makes this wine se...,"{\n ""Region"": []\n}"
2,"A little simple and easy, but there's a wealth...","{\n ""Region"": []\n}"
3,From a dry-farmed vineyard and treated to wild...,"{\n ""Region"": []\n}"
4,"From a site near Annapolis, this wine shows a ...","{\n ""Region"": [""Annapolis""]\n}"


In [156]:

def clean_and_parse_json(string):
    try:
        # # Step 1: Filter out non-JSON responses (like API errors)
        # if not string.strip().startswith('{') or not string.strip().endswith('}'):
        #     raise ValueError("Not a JSON string")
        # Remove triple backticks and language specifier (```json)
        json_compliant_str = string.replace("```", '')
        json_compliant_str = json_compliant_str.replace("json", '')

        # Step 2: Replace single quotes with double quotes
        #json_compliant_str = json_compliant_str.replace("'", '"')

        # # Step 3: Remove trailing commas before closing braces or brackets
        # json_compliant_str = re.sub(r',\s*([}\]])', r'\1', json_compliant_str)

        # # Step 4: Remove leading/trailing whitespace
        # json_compliant_str = json_compliant_str.strip()

        # # Step 5: Parse the cleaned string as JSON
        return json.loads(json_compliant_str)

    except (json.JSONDecodeError, ValueError) as e:
        # For debugging: print problematic strings
        print(f"JSON parsing failed: {e}")
        print(f"Problematic string: {json_compliant_str}")
        return None  # Return None if parsing fails

# Apply the cleaning and parsing function
def_ner['locations_dict'] = def_ner['extracted_locations'].apply(clean_and_parse_json)

# Display successfully parsed attributes
def_ner[['extracted_locations', 'locations_dict']]

,extracted_locations,locations_dict
0,"{\n ""Region"": [""Dry Creek""]\n}",{'Region': ['Dry Creek']}
1,"{\n ""Region"": []\n}",{'Region': []}
2,"{\n ""Region"": []\n}",{'Region': []}
3,"{\n ""Region"": []\n}",{'Region': []}
4,"{\n ""Region"": [""Annapolis""]\n}",{'Region': ['Annapolis']}
...,...,...
995,"{\n ""Region"": [""Serein River""]\n}",{'Region': ['Serein River']}
996,"{\n ""Region"": []\n}",{'Region': []}
997,"{\n ""Region"": [""Alexander Valley""]\n}",{'Region': ['Alexander Valley']}
998,"{\n ""Region"": [""Tuscany""]\n}",{'Region': ['Tuscany']}


In [157]:
non_empty_locations = def_ner[def_ner['locations_dict'].apply(lambda x: bool(x))]

# Display the filtered rows
non_empty_locations

,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,extracted_locations,locations_dict
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.500000,0.246407,0.335451,1.000000,"{\n ""Region"": [""Dry Creek""]\n}",{'Region': ['Dry Creek']}
1,France,A very herbaceous character makes this wine se...,0.277032,0.222222,0.293354,0.272456,0.076923,"{\n ""Region"": []\n}",{'Region': []}
2,US,"A little simple and easy, but there's a wealth...",0.265230,0.222222,0.309569,0.260956,0.740385,"{\n ""Region"": []\n}",{'Region': []}
3,US,From a dry-farmed vineyard and treated to wild...,0.424722,0.444444,0.155139,0.426811,0.586538,"{\n ""Region"": []\n}",{'Region': []}
4,US,"From a site near Annapolis, this wine shows a ...",0.537366,0.611111,0.092931,0.544124,0.634615,"{\n ""Region"": [""Annapolis""]\n}",{'Region': ['Annapolis']}
...,...,...,...,...,...,...,...,...,...
995,France,This is made using a selection of fruit that's...,0.405799,0.388889,0.167604,0.405806,0.144231,"{\n ""Region"": [""Serein River""]\n}",{'Region': ['Serein River']}
996,Spain,"Quite green and also raisiny, which doesn't re...",0.252789,0.111111,0.319435,0.243433,0.692308,"{\n ""Region"": []\n}",{'Region': []}
997,US,Mountain fruit is the key to understanding thi...,0.609966,0.722222,0.065524,0.619769,0.115385,"{\n ""Region"": [""Alexander Valley""]\n}",{'Region': ['Alexander Valley']}
998,Italy,Hats off to the Zonin family for being among t...,0.210837,0.444444,0.415611,0.218393,0.961538,"{\n ""Region"": [""Tuscany""]\n}",{'Region': ['Tuscany']}


In [158]:
unique_locations = set()
def_ner['locations_dict'].apply(lambda x: unique_locations.update(x.keys()) if x else None)

print("Unique locations Found:", unique_locations)

Unique locations Found: {'Vineyard', 'Region', 'Vintage', 'Year', 'Varietal', 'Location', 'Appellation', 'Country', 'Climate'}


In [48]:
import warnings

# Suppress all warnings globally
warnings.filterwarnings('ignore')

In [161]:
list_attributes = ['Vineyard', 'Region', 'Vintage', 'Year', 'Varietal', 'Location', 'Appellation', 'Country', 'Climate']


all_items = {}
for attribute in list_attributes:
    items = set()
    def_ner['locations_dict'].apply(lambda x: items.update(x.get(attribute, [])) if isinstance(x.get(attribute), list) else None)
    all_items[attribute] = items

encoded_dfs = []
for attribute, items in all_items.items():
    temp_df = pd.DataFrame()
    for item in items:
        col_name = f'{attribute}_{item.replace(" ", "_").lower()}'
        temp_df[col_name] = def_ner['locations_dict'].apply(lambda x: int(item in x.get(attribute, [])) if isinstance(x.get(attribute), list) else 0)
    encoded_dfs.append(temp_df)

df_ner_encoded = pd.concat([def_ner] + encoded_dfs, axis=1)

df_ner_encoded.head()

,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,extracted_locations,locations_dict,Vineyard_mercer's_spice_cabinet_vineyard,Vineyard_belle_terre,Vineyard_esteemed_vineyard,Vineyard_robert_young,Vineyard_les_collines_vineyard,Region_new_world,Region_monterey_county,Region_hermitage,Region_american_oak,Region_lake_county,Region_lonesome_spring_ranch,Region_chablis_premier_cru,Region_italy,Region_alsace,Region_hot_climate,Region_elkton_ava,Region_coastal,Region_boushey_vineyard,Region_washington,Region_red_mountain,Region_mayacamas,Region_friuli,Region_cherry_grove,Region_spain,Region_toro,Region_lonesome_springs,Region_mount_st._helena,Region_new_york,Region_meursault,Region_peachy_canyon,Region_languedoc,Region_maremma,Region_georgia,Region_central_coast,Region_charmes,Region_ste._michelle,Region_minnick,Region_oakville,Region_gaillac,Region_côte_du_py,Region_minervois_la_liviniere,Region_central_italy,Region_celilo,Region_bacchus_vineyard,Region_new_zealand,Region_morey_saint-denis,Region_las_amigas,Region_rattlesnake_hills_ava,Region_bordeaux,Region_morgeot_premier_cru,Region_montsant,Region_mount_veeder,Region_moulis,Region_brazil,Region_estate,Region_scansano,Region_howell_mountain,Region_rivesaltes,Region_côte_de_rouffach,Region_northern_italy,Region_paso_robles,Region_cool_growing_region,Region_dwelley,Region_treiso,Region_carneros,Region_california,Region_yakima_valley,Region_serein_river,Region_atlas_peak,Region_tuscany,Region_asti,Region_los_olivos_district,Region_figeac,Region_barolo,Region_beuna_vista,Region_columbia_valley,Region_shenandoah,Region_warm-climate,Region_rutherford,Region_cava,Region_france,Region_rhône,Region_elerding,Region_seven_hills,Region_lazio,Region_southern_italy,Region_candy_mountain_vineyard,Region_cannubi,Region_santa_barbara,Region_coastal_tuscany,Region_boushey,Region_monterey,Region_goldeneye,Region_côtes_du_rhône,Region_lodi,Region_pauillac,Region_napa_valley,Region_puglia,Region_serralunga_d'alba,Region_soave,Region_abruzzo,Region_valdeorras,Region_yolo_county,Region_frassinello,Region_conner_lee_vineyard,Region_calaveras,Region_volnay,Region_renoir,Region_dubrul_vineyard,Region_south_africa,Region_cahors,Region_alexander_valley,Region_santa_lucia_highlands,Region_stagecoach_vineyard,Region_avola,Region_esola_vineyard,Region_champagne,Region_chelan,Region_lirac,Region_san_lucas_ava,Region_southwest,Region_brunate,Region_margaux,Region_st._helena,Region_saint-joseph,Region_albinea,Region_santa_rita,Region_avellino,Region_yountville,Region_bermuda,Region_oregon,Region_temecula_valley,Region_sonoma,Region_châteauneuf,Region_arroyo_seco,Region_walla_walla,Region_epineuil,Region_piedmont,Region_columbia_crest,Region_appellation,Region_sierra_foothills,Region_livermore,Region_amador_foothill,Region_sweetwater,Region_rioja,Region_paso,Region_cartizze,Region_alba,Region_willamette_valley,Region_morrison_lane,Region_foothills,Region_cambria,Region_russian_river_valley,Region_prosecco,Region_coombsville,Region_valpolicella,Region_rattlesnake_hills,Region_walla_walla_valley,Region_umbria,Region_talley_rincon,Region_cool-climate,Region_gascony,Region_côtes_du_rhône_villages,Region_northeast_italy,Region_emilia-romagna,Region_coast,Region_provence,Region_chablis,Region_penedès,Region_sicily,Region_wahluke_slope,Region_anderson_valley,Region_chianti_classico,Region_salinas_valley,Region_ken_wright,Region_montalcino,Region_dry_creek,Region_arroyo_seco_ava,Region_warm_climate,Region_annapolis,Region_beaujolais,Region_sonoma_county,Region_ciel's_vineyard,Region_mountain,Region_diamond_mountain,Region_dalton_vineyard,Region_st._emilion,Region_santa_ynez_valley,Region_russian_river,Region_birch_creek_vineyard,Region_barbaresco,Region_champoux,Region_livermore_valley,Region_southern_france,Region_ribera_del_duero,Region_greenwood_ridge,Region_cliff_creek,Region_campobello,Region_burgundy,Region_horse_heaven_hills,Region_chambolle-musigny,Region_

In [162]:
zero_proportion = (df_ner_encoded == 0).sum() / len(df)

# Step 2: Filter out columns with more than 99% zeros
df_ner_filtered = df_ner_encoded.loc[:, zero_proportion <= 0.99]
df_ner_filtered

,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,extracted_locations,locations_dict,Region_bordeaux,Region_california,Region_napa
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.500000,0.246407,0.335451,1.000000,"{\n ""Region"": [""Dry Creek""]\n}",{'Region': ['Dry Creek']},0,0,0
1,France,A very herbaceous character makes this wine se...,0.277032,0.222222,0.293354,0.272456,0.076923,"{\n ""Region"": []\n}",{'Region': []},0,0,0
2,US,"A little simple and easy, but there's a wealth...",0.265230,0.222222,0.309569,0.260956,0.740385,"{\n ""Region"": []\n}",{'Region': []},0,0,0
3,US,From a dry-farmed vineyard and treated to wild...,0.424722,0.444444,0.155139,0.426811,0.586538,"{\n ""Region"": []\n}",{'Region': []},0,0,0
4,US,"From a site near Annapolis, this wine shows a ...",0.537366,0.611111,0.092931,0.544124,0.634615,"{\n ""Region"": [""Annapolis""]\n}",{'Region': ['Annapolis']},0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
995,France,This is made using a selection of fruit that's...,0.405799,0.388889,0.167604,0.405806,0.144231,"{\n ""Region"": [""Serein River""]\n}",{'Region': ['Serein River']},0,0,0
996,Spain,"Quite green and also raisiny, which doesn't re...",0.252789,0.111111,0.319435,0.243433,0.692308,"{\n ""Region"": []\n}",{'Region': []},0,0,0
997,US,Mountain fruit is the key to understanding thi...,0.609966,0.722222,0.065524,0.619769,0.115385,"{\n ""Region"": [""Alexander Valley""]\n}",{'Region': ['Alexander Valley']},0,0,0
998,Italy,Hats off to the Zonin family for being among t...,0.210837,0.444444,0.415611,0.218393,0.961538,"{\n ""Region"": [""Tuscany""]\n}",{'Region': ['Tuscany']},0,0,0


In [163]:
df_ner_filtered=df_ner_filtered.drop('extracted_locations', axis=1)
df_ner_filtered=df_ner_filtered.drop('locations_dict', axis=1)

## LLM summarisation with TF-IDF

We can now request the api to provide us with summaries of the description. This would later work well with TF-IDF function to make ambeddings for concise sentences

In [63]:
def summarize_description(description, retries=3, wait=5):
    headers = {
        'Content-Type': 'application/json',
        'x-api-key': API_KEY
    }

    payload = {
        "model": "gpt-4o-mini",
        "messages": [
            {
                "role": "system",
                "content": (
                    "Summarize this wine description into one concise sentence. "
                    "Return the result in JSON format: {\"Summary\": \"\"}."
                )
            },
            {"role": "user", "content": f"Description: {description}"}
        ],
        "temperature": 0.5
    }

    for attempt in range(retries):
        response = requests.post(API_URL, headers=headers, json=payload)
        
        if response.status_code == 200:
            # Return the raw JSON content as string
            result = response.json()
            raw_json_string = result['choices'][0]['message']['content'].strip()
            return raw_json_string
        
        elif response.status_code in [429, 504]:  # Handle rate limits or timeouts
            print(f"Error {response.status_code} on attempt {attempt + 1}. Retrying in {wait} seconds...")
            time.sleep(wait)
        else:
            print(f"Unexpected error {response.status_code}: {response.text}")
            return None

    return "Error: Max retries exceeded"

df_eda = pd.read_csv("data/wine_eda_features.csv")
df_summary= df_eda.copy(deep=True)

df_summary['summary'] = df_eda['description'].apply(summarize_description)

# Display the DataFrame with summaries
df_summary[['description', 'summary']].head()

Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt 1. Retrying in 5 seconds...
Error 429 on attempt

,description,summary
0,"Very good Dry Creek Zin, robust and dry and sp...","{""Summary"": ""A robust and dry Dry Creek Zinfan..."
1,A very herbaceous character makes this wine se...,"{""Summary"": ""This lightweight wine features a ..."
2,"A little simple and easy, but there's a wealth...","{""Summary"": ""This delicate wine features raspb..."
3,From a dry-farmed vineyard and treated to wild...,"{""Summary"": ""This dry-farmed Petite wine featu..."
4,"From a site near Annapolis, this wine shows a ...","{""Summary"": ""This light-bodied wine from near ..."


In [65]:
def clean_and_parse_json(json_compliant_str):
    try:
        # # Step 1: Filter out non-JSON responses (like API errors)
        # if not string.strip().startswith('{') or not string.strip().endswith('}'):
        #     raise ValueError("Not a JSON string")
        # Remove triple backticks and language specifier (```json)
        #json_compliant_str = string.replace("```", '')
        #json_compliant_str = json_compliant_str.replace("json", '')

        # Step 2: Replace single quotes with double quotes
        #json_compliant_str = json_compliant_str.replace("'", '"')

        # # Step 3: Remove trailing commas before closing braces or brackets
        # json_compliant_str = re.sub(r',\s*([}\]])', r'\1', json_compliant_str)

        # # Step 4: Remove leading/trailing whitespace
        # json_compliant_str = json_compliant_str.strip()

        # # Step 5: Parse the cleaned string as JSON
        return json.loads(json_compliant_str)

    except (json.JSONDecodeError, ValueError) as e:
        # For debugging: print problematic strings
        print(f"JSON parsing failed: {e}")
        print(f"Problematic string: {json_compliant_str}")
        return None  # Return None if parsing fails

# Apply the cleaning and parsing function
df_summary['summary_dict'] = df_summary['summary'].apply(clean_and_parse_json)

# Display successfully parsed attributes
df_summary[['summary', 'summary_dict']].head(2)

,summary,summary_dict
0,"{""Summary"": ""A robust and dry Dry Creek Zinfan...",{'Summary': 'A robust and dry Dry Creek Zinfan...
1,"{""Summary"": ""This lightweight wine features a ...",{'Summary': 'This lightweight wine features a ...


In [68]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Generate TF-IDF embeddings for the summaries
vectorizer = TfidfVectorizer(max_features=30, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df_summary['summary'])

# Convert TF-IDF matrix to DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# Concatenate embeddings with original DataFrame
df_final = pd.concat([df_summary, tfidf_df], axis=1)

# Display the final DataFrame with embeddings
df_final.head(2)


,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,summary,summary_dict,acidity,...,oak,palate,red,rich,ripe,spice,summary,sweet,tannins,wine
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.500000,0.246407,0.335451,104,"{""Summary"": ""A robust and dry Dry Creek Zinfan...",{'Summary': 'A robust and dry Dry Creek Zinfan...,0.0,...,0.0,0.0,0.447754,0.0,0.0,0.0,0.138370,0.0,0.0,0.000000
1,France,A very herbaceous character makes this wine se...,0.277032,0.222222,0.293354,0.272456,8,"{""Summary"": ""This lightweight wine features a ...",{'Summary': 'This lightweight wine features a ...,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.244052,0.0,0.0,0.362542


## Finilizing the dataset

Let's now check our created datasets and combine the features for the model

In [141]:
df_llm.head(1)

,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.5,0.246407,0.335451,1.0


In [164]:
df_ner_filtered.head(1)

,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,Region_bordeaux,Region_california,Region_napa
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.5,0.246407,0.335451,1.0,0,0,0


In [80]:
df_llm=df_llm.drop('description', axis=1)

In [95]:
df_llm=df_llm.drop('extracted_locations', axis=1)
df_llm=df_llm.drop('locations_dict', axis=1)
df_llm.head(1)

,country,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,Vineyard_Encoded,Country_Encoded,Other Locations_Encoded,Drink Until_Encoded,Winery_Encoded,Varietals_Encoded,Adjacent Vineyards_Encoded,Location_Encoded,Importer_Encoded,State_Encoded,Region_Encoded,Imported By_Encoded,Vintage_Encoded,Varietal_Encoded,Other_Location_Encoded,Location Type_Encoded,Year_Encoded,Appellation_Encoded,Climate_Encoded
0,US,0.328377,0.5,0.246407,0.335451,104,112,11,4,1,1,1,1,1,1,1,47,5,1,2,1,3,1,4,1


In [138]:
df.head(1)

,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,Ageability_0,Ageability_1,Ageability_2,Ageability_3,Acidity_0,Acidity_1,Acidity_2,Acidity_3,Acidity_4,Sweetness_0,Sweetness_1,Sweetness_2,Sweetness_3,Sweetness_4,Sweetness_5,Sweetness_6,Sweetness_7,Sweetness_8,Sweetness_9,Sweetness_10,Sweetness_11,Body_0,Body_1,Body_2,Body_3,Body_4,Body_5,Body_6,Tannins_0,Tannins_1,Tannins_2,Tannins_3,Tannins_4,Finish_0,Finish_1,Finish_2,Finish_3,Finish_4,Finish_5,Finish_6,Finish_7,Finish_8,Finish_9,Flavor_embed_0,Flavor_embed_1,Flavor_embed_2,Flavor_embed_3,Flavor_embed_4,Flavor_embed_5,Flavor_embed_6,Flavor_embed_7,Flavor_embed_8,Flavor_embed_9,Flavor_embed_10,Flavor_embed_11,Flavor_embed_12,Flavor_embed_13,Flavor_embed_14,Flavor_embed_15,Flavor_embed_16,Flavor_embed_17,Flavor_embed_18,Flavor_embed_19,Flavor_embed_20,Flavor_embed_21,Flavor_embed_22,Flavor_embed_23,Flavor_embed_24,Flavor_embed_25,Flavor_embed_26,Flavor_embed_27,Flavor_embed_28,Flavor_embed_29,Flavor_embed_30,Flavor_embed_31,Flavor_embed_32,Flavor_embed_33,Flavor_embed_34,Flavor_embed_35,Flavor_embed_36,Flavor_embed_37,Flavor_embed_38,Flavor_embed_39,Flavor_embed_40,Flavor_embed_41,Flavor_embed_42,Flavor_embed_43,Flavor_embed_44,Flavor_embed_45,Flavor_embed_46,Flavor_embed_47,Flavor_embed_48,Flavor_embed_49
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.5,0.246407,0.335451,1.0,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,0.002942,-0.018224,-0.00811,-0.004448,0.005891,0.0053,-0.006271,0.024145,-0.025567,0.000539,-0.013274,0.01682,0.002571,0.014558,0.006142,-0.007492,-0.002435,-0.025775,-0.011514,-0.005155,-0.001678,0.010637,0.009606,0.00179,0.014124,0.002587,-0.003902,-0.001312,-0.003493,-0.015485,-0.002237,-0.008857,-0.009433,0.008766,-0.007805,-0.008802,0.002012,-0.037874,-0.006663,0.013167,-0.010712,0.006425,0.024395,-0.023538,0.001811,-0.003636,0.002449,0.013644,0.021846,-0.009787


In [139]:
df=df.drop('description', axis=1)

In [92]:
df_final.head(1)

,country,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,acidity,aging,aromas,balanced,berry,black,blackberry,blend,cabernet,cherry,complemented,dry,features,featuring,finish,flavors,fresh,fruit,making,notes,oak,palate,red,rich,ripe,spice,sweet,tannins,wine
0,US,0.328377,0.5,0.246407,0.335451,104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.338891,0.0,0.787645,0.0,0.0,0.0,0.212458,0.0,0.0,0.0,0.0,0.0,0.0,0.447754,0.0,0.0,0.0,0.0,0.0,0.0


In [87]:
df_final=df_final.drop('description', axis=1)

In [91]:
df_final=df_final.drop('summary', axis=1)
df_final=df_final.drop('summary_dict', axis=1)
df_final.head(1)

,country,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,acidity,aging,aromas,balanced,berry,black,blackberry,blend,cabernet,cherry,complemented,dry,features,featuring,finish,flavors,fresh,fruit,making,notes,oak,palate,red,rich,ripe,spice,sweet,tannins,wine
0,US,0.328377,0.5,0.246407,0.335451,104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.338891,0.0,0.787645,0.0,0.0,0.0,0.212458,0.0,0.0,0.0,0.0,0.0,0.0,0.447754,0.0,0.0,0.0,0.0,0.0,0.0


In [166]:
exclude_cols = ['country','log_price_normalized', 'points_normalized', 'vfm_normalized', 'log_pxp_normalized', 'encoded_variety']
df_llm_final = pd.concat([df_llm, df[[col for col in df.columns if col not in exclude_cols]],df_final[[col for col in df_final.columns if col not in exclude_cols]], df_ner_filtered[[col for col in df_ner_filtered.columns if col not in exclude_cols]]], axis=1)
df_llm_final.head(1)

,country,description,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,Ageability_0,Ageability_1,Ageability_2,Ageability_3,Acidity_0,Acidity_1,Acidity_2,Acidity_3,Acidity_4,Sweetness_0,Sweetness_1,Sweetness_2,Sweetness_3,Sweetness_4,Sweetness_5,Sweetness_6,Sweetness_7,Sweetness_8,Sweetness_9,Sweetness_10,Sweetness_11,Body_0,Body_1,Body_2,Body_3,Body_4,Body_5,Body_6,Tannins_0,Tannins_1,Tannins_2,Tannins_3,Tannins_4,Finish_0,Finish_1,Finish_2,Finish_3,Finish_4,Finish_5,Finish_6,Finish_7,Finish_8,Finish_9,Flavor_embed_0,Flavor_embed_1,Flavor_embed_2,Flavor_embed_3,Flavor_embed_4,Flavor_embed_5,Flavor_embed_6,Flavor_embed_7,Flavor_embed_8,Flavor_embed_9,Flavor_embed_10,Flavor_embed_11,Flavor_embed_12,Flavor_embed_13,Flavor_embed_14,Flavor_embed_15,Flavor_embed_16,Flavor_embed_17,Flavor_embed_18,Flavor_embed_19,Flavor_embed_20,Flavor_embed_21,Flavor_embed_22,Flavor_embed_23,Flavor_embed_24,Flavor_embed_25,Flavor_embed_26,Flavor_embed_27,Flavor_embed_28,Flavor_embed_29,Flavor_embed_30,Flavor_embed_31,Flavor_embed_32,Flavor_embed_33,Flavor_embed_34,Flavor_embed_35,Flavor_embed_36,Flavor_embed_37,Flavor_embed_38,Flavor_embed_39,Flavor_embed_40,Flavor_embed_41,Flavor_embed_42,Flavor_embed_43,Flavor_embed_44,Flavor_embed_45,Flavor_embed_46,Flavor_embed_47,Flavor_embed_48,Flavor_embed_49,acidity,aging,aromas,balanced,berry,black,blackberry,blend,cabernet,cherry,complemented,dry,features,featuring,finish,flavors,fresh,fruit,making,notes,oak,palate,red,rich,ripe,spice,sweet,tannins,wine,description,Region_bordeaux,Region_california,Region_napa
0,US,"Very good Dry Creek Zin, robust and dry and sp...",0.328377,0.5,0.246407,0.335451,1.0,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,0.002942,-0.018224,-0.00811,-0.004448,0.005891,0.0053,-0.006271,0.024145,-0.025567,0.000539,-0.013274,0.01682,0.002571,0.014558,0.006142,-0.007492,-0.002435,-0.025775,-0.011514,-0.005155,-0.001678,0.010637,0.009606,0.00179,0.014124,0.002587,-0.003902,-0.001312,-0.003493,-0.015485,-0.002237,-0.008857,-0.009433,0.008766,-0.007805,-0.008802,0.002012,-0.037874,-0.006663,0.013167,-0.010712,0.006425,0.024395,-0.023538,0.001811,-0.003636,0.002449,0.013644,0.021846,-0.009787,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.338891,0.0,0.787645,0.0,0.0,0.0,0.212458,0.0,0.0,0.0,0.0,0.0,0.0,0.447754,0.0,0.0,0.0,0.0,0.0,0.0,"Very good Dry Creek Zin, robust and dry and sp...",0,0,0


In [167]:
df_llm_final=df_llm_final.drop('description', axis=1)

In [168]:
df_llm_final.head(1)

,country,log_price_normalized,points_normalized,vfm_normalized,log_pxp_normalized,encoded_variety,Ageability_0,Ageability_1,Ageability_2,Ageability_3,Acidity_0,Acidity_1,Acidity_2,Acidity_3,Acidity_4,Sweetness_0,Sweetness_1,Sweetness_2,Sweetness_3,Sweetness_4,Sweetness_5,Sweetness_6,Sweetness_7,Sweetness_8,Sweetness_9,Sweetness_10,Sweetness_11,Body_0,Body_1,Body_2,Body_3,Body_4,Body_5,Body_6,Tannins_0,Tannins_1,Tannins_2,Tannins_3,Tannins_4,Finish_0,Finish_1,Finish_2,Finish_3,Finish_4,Finish_5,Finish_6,Finish_7,Finish_8,Finish_9,Flavor_embed_0,Flavor_embed_1,Flavor_embed_2,Flavor_embed_3,Flavor_embed_4,Flavor_embed_5,Flavor_embed_6,Flavor_embed_7,Flavor_embed_8,Flavor_embed_9,Flavor_embed_10,Flavor_embed_11,Flavor_embed_12,Flavor_embed_13,Flavor_embed_14,Flavor_embed_15,Flavor_embed_16,Flavor_embed_17,Flavor_embed_18,Flavor_embed_19,Flavor_embed_20,Flavor_embed_21,Flavor_embed_22,Flavor_embed_23,Flavor_embed_24,Flavor_embed_25,Flavor_embed_26,Flavor_embed_27,Flavor_embed_28,Flavor_embed_29,Flavor_embed_30,Flavor_embed_31,Flavor_embed_32,Flavor_embed_33,Flavor_embed_34,Flavor_embed_35,Flavor_embed_36,Flavor_embed_37,Flavor_embed_38,Flavor_embed_39,Flavor_embed_40,Flavor_embed_41,Flavor_embed_42,Flavor_embed_43,Flavor_embed_44,Flavor_embed_45,Flavor_embed_46,Flavor_embed_47,Flavor_embed_48,Flavor_embed_49,acidity,aging,aromas,balanced,berry,black,blackberry,blend,cabernet,cherry,complemented,dry,features,featuring,finish,flavors,fresh,fruit,making,notes,oak,palate,red,rich,ripe,spice,sweet,tannins,wine,Region_bordeaux,Region_california,Region_napa
0,US,0.328377,0.5,0.246407,0.335451,1.0,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,0.002942,-0.018224,-0.00811,-0.004448,0.005891,0.0053,-0.006271,0.024145,-0.025567,0.000539,-0.013274,0.01682,0.002571,0.014558,0.006142,-0.007492,-0.002435,-0.025775,-0.011514,-0.005155,-0.001678,0.010637,0.009606,0.00179,0.014124,0.002587,-0.003902,-0.001312,-0.003493,-0.015485,-0.002237,-0.008857,-0.009433,0.008766,-0.007805,-0.008802,0.002012,-0.037874,-0.006663,0.013167,-0.010712,0.006425,0.024395,-0.023538,0.001811,-0.003636,0.002449,0.013644,0.021846,-0.009787,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.338891,0.0,0.787645,0.0,0.0,0.0,0.212458,0.0,0.0,0.0,0.0,0.0,0.0,0.447754,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0


In [169]:
df_llm_final.to_csv('data/wine_llm.csv', index=False)